# Introduction and Preparation
- [Project Introduction](#project-introduction)
- [Datasets](#datasets)
- [Merging Datasets](#merging-datasets)
    - [Preparation](#preparation)
    - [Merge](#merge)
        - [Raw Merge](#raw-merge)
        - [Normalized Merge](#normalized-merge)
        - [Remaining Data](#remaining-data)
    - [Saving to Dataframe](#saving-dataset)


## Project Introduction

Streaming platforms like Spotify have seen a dramatic rise in annual sales volume, increasing nearly sevenfold over the past decade¹. This growth is due in part to advanced recommendation systems, which rely not only on user behavior but also on detailed analysis of the music itself. One particularly rich source of information is the song lyrics.

This project explores the relationship between Spotify song lyrics and their corresponding musical attributes, such as popularity, danceability, energy, loudness, speechiness, valence, and tempo. We also investigate how well generative models can produce lyrics conditioned on a specific genre.

Our motivation is to examine how emotional, structural, and stylistic elements in lyrics relate to the perceived or measured characteristics of a track. We aim to understand to what extent lyrical content can reflect or influence a song’s musical qualities.

A potential application of this work is the enhancement of music recommendation systems on platforms like Spotify or Apple Music. By incorporating lyrical analysis, such systems can suggest songs based not only on acoustic features but also on the emotional and thematic content of the lyrics, even across different genres. A hypothetical client for this project could be a music streaming service aiming to improve its recommendation algorithms through a deeper understanding of lyrics.

In this context, we aim to address the following research questions:
- What correlations exist between lyrical features and musical attributes? Can we accurately predict characteristics such as energy or danceability using only song lyrics?
- Is it possible to generate song lyrics based on a given genre, and how effectively do different generative models perform in this task?

¹ Statista. (2024). Umsätze mit Musikstreaming-Aboservices weltweit in den Jahren 2010 bis 2024 (in Milliarden US-Dollar)[Graph]. In Spotify - Statistik-Report zu Spotify  (p. 8). Statista. https://de.statista.com/statistik/studie/id/21276/dokument/spotify/


## Datasets

We selected two partially overlapping datasets and aim to merge them to create a unified dataset that combines the strengths of both sources.
- [Spotify One Million Tracks](https://www.kaggle.com/datasets/amitanshjoshi/spotify-1million-tracks): 1159764 songs, 19 columns
- [Genius Song Lyrics with Language information](https://www.kaggle.com/datasets/carlosgdcj/genius-song-lyrics-with-language-information): 5134856 songs, 11 columns

In the best-case scenario, where every song in the Spotify dataset has a corresponding entry in the Genius dataset, we would obtain a combined dataset of approximately one million songs. This merged dataset would offer around 29 features for further analysis.

## Methods

To address our main research questions, we begin by constructing a unified dataset that combines lyrical and musical information. This involves merging two complementary sources by aligning entries based on artist names and track titles. Since both datasets use different naming conventions and formatting, we implement normalization steps to handle inconsistencies, such as variations in capitalization, punctuation, or numerical suffixes.

Following the merge, we clean the dataset to filter out low-quality or incomplete entries. This includes removing songs with missing or non-English lyrics, unusually short texts, and duplicate records. The goal is to create a high-quality corpus that accurately represents the relationship between lyrical content and musical attributes. This preprocessed dataset then serves as the foundation for all subsequent analysis and model training.

## Merging Datasets
### Preparation

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
import regex as re
import os
import pickle

In [ ]:
file_path = "spotify_data.csv"

# Load the latest version
spotify_df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "amitanshjoshi/spotify-1million-tracks",
  file_path,
)

# remove "Unnamed: 0" column since original index is not needed, use dataframe index instead
spotify_df.drop(columns=["Unnamed: 0"], inplace=True)

spotify_df.columns

/tmp/ipykernel_580998/2898874327.py:4: DeprecationWarning: load_dataset is deprecated and will be removed in future version.
  spotify_df = kagglehub.load_dataset(


Index(['artist_name', 'track_name', 'track_id', 'popularity', 'year', 'genre',
       'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
       'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
       'duration_ms', 'time_signature'],
      dtype='object')

In [ ]:
file_path = "song_lyrics.csv"

# Load the latest version
genius_df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "carlosgdcj/genius-song-lyrics-with-language-information",
  file_path,
)

genius_df.columns

/tmp/ipykernel_580998/2964555141.py:4: DeprecationWarning: load_dataset is deprecated and will be removed in future version.
  genius_df = kagglehub.load_dataset(


Index(['title', 'tag', 'artist', 'year', 'views', 'features', 'lyrics', 'id',
       'language_cld3', 'language_ft', 'language'],
      dtype='object')

Renaming the datasets to have matching column names for easier merging.

In [ ]:
genius_df.rename(columns={'title': 'track_name', 'artist': 'artist_name'}, inplace=True)
genius_df

,track_name,tag,artist_name,year,views,features,lyrics,id,language_cld3,language_ft,language
0,Killa Cam,rap,Cam'ron,2004,173166,"{""Cam\\'ron"",""Opera Steve""}","[Chorus: Opera Steve & Cam'ron]\nKilla Cam, Ki...",1,en,en,en
1,Can I Live,rap,JAY-Z,1996,468624,{},"[Produced by Irv Gotti]\n\n[Intro]\nYeah, hah,...",3,en,en,en
2,Forgive Me Father,rap,Fabolous,2003,4743,{},Maybe cause I'm eatin\nAnd these bastards fien...,4,en,en,en
3,Down and Out,rap,Cam'ron,2004,144404,"{""Cam\\'ron"",""Kanye West"",""Syleena Johnson""}",[Produced by Kanye West and Brian Miller]\n\n[...,5,en,en,en
4,Fly In,rap,Lil Wayne,2005,78271,{},"[Intro]\nSo they ask me\n""Young boy\nWhat you ...",6,en,en,en
...,...,...,...,...,...,...,...,...,...,...,...
5134851,Ocean,pop,Effemar,2022,3,{},[Verse 1]\nDance for me now\nKeeping yourself ...,7882842,en,en,en
5134852,64 Bars,rap,Rapido,2022,4,{},"[Intro]\n\nJa, ja\n\n[Part 1]\n\nR-A-H, Merhab...",7882843,de,de,de
5134853,Raise Our Hands,pop,"Culture Code, Pag & Mylo",2016,3,"{Elex,""Culture Code / Pag & Mylo""}",[Verse 1]\nHere our purpose feels alive\nWe ar...,7882845,en,en,en
5134854,CEO,rap,Antropolita,2022,5,{},Jestem CEO w tym\nTo jara twoją bitch\nNikt na...,7882846,pl,pl,pl


In [ ]:
print(spotify_df.isna().sum())
print(genius_df.isna().sum())

artist_name         15
track_name           1
track_id             0
popularity           0
year                 0
genre                0
danceability         0
energy               0
key                  0
loudness             0
mode                 0
speechiness          0
acousticness         0
instrumentalness     0
liveness             0
valence              0
tempo                0
duration_ms          0
time_signature       0
dtype: int64
track_name          188
tag                   0
artist_name           0
year                  0
views                 0
features              0
lyrics                0
id                    0
language_cld3     90966
language_ft      134322
language         226918
dtype: int64


In [ ]:
spotify_df.dropna(subset=['artist_name','track_name'], inplace=True)
genius_df.dropna(subset=['artist_name','track_name'], inplace=True)

### Merge

In [ ]:
spotify_df[['artist_name','track_name']] = spotify_df[['artist_name','track_name']].astype(str)
genius_df[['artist_name','track_name']] = genius_df[['artist_name','track_name']].astype(str)

# All column intersections
spotify_df.columns.intersection(genius_df.columns)

Index(['artist_name', 'track_name', 'year'], dtype='object')

Because we don't only have artist_name and track_name as columns which are present in both datasets but also year, we have to set a suffix while merging.

#### Raw Merge
This form of merge only takes the raw data (thus we are calling it raw merge) into account without any normalization before.

In [ ]:
merge_df = pd.merge(spotify_df, genius_df, on=['artist_name', 'track_name'], how='left', indicator='raw_merge', suffixes=('', '_genius'))
raw_merge_df = merge_df[merge_df['raw_merge'] == 'both']
raw_merge_df.columns

Index(['artist_name', 'track_name', 'track_id', 'popularity', 'year', 'genre',
       'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
       'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
       'duration_ms', 'time_signature', 'tag', 'year_genius', 'views',
       'features', 'lyrics', 'id', 'language_cld3', 'language_ft', 'language',
       'raw_merge'],
      dtype='object')

In [ ]:
raw_merge_df.sample(5)

,artist_name,track_name,track_id,popularity,year,genre,danceability,energy,key,loudness,...,tag,year_genius,views,features,lyrics,id,language_cld3,language_ft,language,raw_merge
216732,Sodom,Rolling Thunder,66vwd2w3FrlyeArQUkMxzn,23,2016,black-metal,0.467,0.9790,2,-5.856,...,rock,2016.0,1334.0,{},Getting closer to the gasping prey\nTo learn t...,2851204.0,en,en,en,both
1143239,Tony Bennett,Christmas In Herald Square,4XJ9FZtkJxnzGyB6mVXgld,22,2011,jazz,0.380,0.0873,5,-12.906,...,pop,1998.0,531.0,{},Bundle up from head to toe\nAnd smile as it be...,1358226.0,en,en,en,both
445500,Stevie Dinner,Stray Cat,5mcjcF3rrdwX1pGRqNBy3d,13,2020,garage,0.703,0.5080,9,-7.527,...,pop,2020.0,7.0,{},I wish all the stray cats in my neighborhood\n...,6719383.0,en,en,en,both
333845,Elwood Stray,You Lost,6UOuW9RRhiwOyTgfSoaktV,22,2018,german,0.489,0.9310,2,-4.104,...,rock,2018.0,963.0,"{""Kassim Auale""}",Out of my head\nThat's what I’ve said\nBetween...,3944989.0,en,en,en,both
698743,The Flying Burrito Brothers,Windfall,2UTZys44oUu8nxcS9dvzJL,9,2001,psych-rock,0.571,0.4870,9,-12.435,...,pop,1997.0,28.0,{},Now and then it keeps you running\nIt never se...,1112252.0,en,en,en,both


In [ ]:
merge_df['raw_merge'].value_counts()

raw_merge
left_only     927775
both          231973
right_only         0
Name: count, dtype: int64

As we can see this merge brought us around 230k songs that are present in both datasets.

#### Normalized Merge
In contrast to normalized merge, full merge will also compare both relevant columns but first apply a normalization to hopefully find more matches where e.g. some community member from genius placed an apostrophe where spotify didn't.

First we gather all entries from the spotify dataset that couln't be merged in the raw merge.

In [ ]:
raw_unmatched_spotify_df = merge_df[merge_df['raw_merge'] == 'left_only'].copy()

# Remove every NaN column from the missing_spotify_df to avoid unnecessary columns in the merge
raw_unmatched_spotify_df = raw_unmatched_spotify_df[raw_unmatched_spotify_df.columns.intersection(spotify_df.columns)]


print(raw_unmatched_spotify_df.columns)
print(raw_unmatched_spotify_df.shape)

raw_unmatched_spotify_df.sample(5)

Index(['artist_name', 'track_name', 'track_id', 'popularity', 'year', 'genre',
       'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
       'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
       'duration_ms', 'time_signature'],
      dtype='object')
(927775, 19)


,artist_name,track_name,track_id,popularity,year,genre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature
28811,Andrei Krylov,Ancient Spirit of Warhammer in Marienburg Cath...,1TC0oH9ew8UzmgTMTyWuBu,8,2012,guitar,0.515,0.287,5,-14.122,0,0.0355,0.972000,0.752000,0.1370,0.314,78.669,145322,4
917691,Bobby Vinton,Here In My Heart,0PsuMG7QIm8PqqLBjyrK2M,6,2006,rock-n-roll,0.253,0.413,11,-9.388,1,0.0314,0.266000,0.000002,0.2820,0.181,121.819,167387,3
841411,The Chemical Brothers,Believe - Radio Edit,4mZ2NLuYH84s5mo8jEZ533,14,2005,breakbeat,0.679,0.756,0,-4.244,1,0.0496,0.000367,0.013900,0.0694,0.786,131.979,231613,4
454994,mxmtoon,almost home,1B4XVCWbQcMQ1wFMLrjjOz,48,2020,indie-pop,0.554,0.483,5,-5.936,1,0.0262,0.299000,0.000001,0.3270,0.423,138.034,208813,3
1022645,Chris Zabriskie,Prelude No. 14,2cD49D1j3ZxJBpWfEpst16,7,2009,ambient,0.151,0.234,8,-22.308,1,0.0453,0.986000,0.961000,0.1220,0.237,107.265,58900,4


In [ ]:
def normalize(text):
    return re.sub('[^a-zA-Z]+', '', text).lower()

In [ ]:
# Apply normalization to both DataFrames
raw_unmatched_spotify_df['artist_name_norm'] = raw_unmatched_spotify_df['artist_name'].apply(normalize)
raw_unmatched_spotify_df['track_name_norm'] = raw_unmatched_spotify_df['track_name'].apply(normalize)

genius_df['artist_name_norm'] = genius_df['artist_name'].apply(normalize)
genius_df['track_name_norm'] = genius_df['track_name'].apply(normalize)

# Since removing all non-alphabetic characters may lead to empty strings, we need to check for that and remove them
# Some of these entries for example had titles written in chinese or cyrillic alphabet
entry_count_before = raw_unmatched_spotify_df.shape[0]
unmatched_notnan = raw_unmatched_spotify_df[(raw_unmatched_spotify_df['artist_name_norm'] != "") & (raw_unmatched_spotify_df['track_name_norm'] != "")]

normalized_merge_nan_removal_count = entry_count_before - unmatched_notnan.shape[0]
print('Removed', normalized_merge_nan_removal_count, ' entries out of', unmatched_notnan.shape[0])

Removed 29077  entries out of 898698


As we can see all artist and track names are now normalized:

In [ ]:
unmatched_notnan.sample(20)[['artist_name', 'track_name', 'artist_name_norm', 'track_name_norm']]

,artist_name,track_name,artist_name_norm,track_name_norm
993613,Bushido,Outro,bushido,outro
433781,Regard,Secrets - MOTi Remix,regard,secretsmotiremix
573534,Frankie Laine,Hold Me,frankielaine,holdme
749697,Orquesta tipica Francisco Canaro,Cuando Vuelva,orquestatipicafranciscocanaro,cuandovuelva
697797,Joe Jackson,Right And Wrong - Live At The Roundabout Theat...,joejackson,rightandwrongliveattheroundabouttheatrenewyork...
671980,Kankick,Love Hardcore - Underground,kankick,lovehardcoreunderground
540173,Bill Bellamy,Will & Jada Entanglement,billbellamy,willjadaentanglement
538143,thuy,insecurities,thuy,insecurities
552304,Mc Paiva ZS,Nois É os Cara,mcpaivazs,noisoscara
554964,Mortemia,Frozen,mortemia,frozen


In [ ]:
# merging now on the normalized columns
merge_df_norm = pd.merge(unmatched_notnan, genius_df, on=['artist_name_norm', 'track_name_norm'], how='left', indicator='normalized_merge', suffixes=('', '_genius'))

merge_df_norm["normalized_merge"].value_counts()

normalized_merge
left_only     767302
both          132963
right_only         0
Name: count, dtype: int64

Here we got around 130k matches in addition to our raw merge dataset.

In [ ]:
# Result of full merge are entries that were in the missing_spotify_df and also in the genius_df
normalized_merge_df = merge_df_norm[merge_df_norm['normalized_merge'] == 'both']
print(normalized_merge_df.shape, normalized_merge_df.columns)
normalized_merge_df.sample(20)[['artist_name_norm', 'track_name_norm', 'artist_name', 'artist_name_genius' , 'track_name', 'track_name_genius', 'normalized_merge', 'lyrics']]

(132963, 33) Index(['artist_name', 'track_name', 'track_id', 'popularity', 'year', 'genre',
       'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
       'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
       'duration_ms', 'time_signature', 'artist_name_norm', 'track_name_norm',
       'track_name_genius', 'tag', 'artist_name_genius', 'year_genius',
       'views', 'features', 'lyrics', 'id', 'language_cld3', 'language_ft',
       'language', 'normalized_merge'],
      dtype='object')


,artist_name_norm,track_name_norm,artist_name,artist_name_genius,track_name,track_name_genius,normalized_merge,lyrics
825579,sharonjonesthedapkings,imnotgonnacry,Sharon Jones & The Dap-Kings,Sharon Jones & The Dap-Kings,I'm Not Gonna Cry,Im Not Gonna Cry,both,"Baby, baby, baby\nI'm tired of tryin' to chang..."
804351,empireempireiwasalonelyestate,withyourgreatestfearsrealizedyouwillnotbecomfo...,empire! empire! (i was a lonely estate),Empire! Empire! (I was A Lonely Estate),"With Your Greatest Fears Realized, You Will No...",With Your Greatest Fears Realized You Will Not...,both,You wore a hand-me-down dress that never fit q...
139341,fromfirsttolast,notetoself,From First To Last,From First to Last,Note To Self,Note to Self,both,[Instrumental Intro]\n\n[Chorus]\nTwo roads sp...
42589,countingcrows,untitledlovesong,Counting Crows,Counting Crows,Untitled (Love Song),Untitled Love Song,both,"If tonight’s the night, then today is my day\n..."
532204,ludmilaferber,ouaetomeposse,Ludmila Ferber,Ludmila Ferber,Ouça e Tome Posse,Ouça E Tome Posse,both,Ouça e tome posse\nDa Palavra do Deus Vivo\nEl...
13736,wisinyandel,algomegustadeti,Wisin & Yandel,Wisin & Yandel,Algo Me Gusta De Ti,Algo Me Gusta de Ti,both,"[Letra de ''Algo Me Gusta De Ti'' ft. T-Pain, ..."
118256,estopa,elrunrundirectoacstico,Estopa,Estopa,El Run Run - Directo Acústico,El Run Run Directo Acústico,both,"[Letra de ""El Run Run (Directo Acústico)""]\n\n..."
537052,hllbnt,repeatpatternssuture,H3llb3nt,H3llb3nt,Repeat Patterns (Suture),Repeat Patterns Suture,both,Repeat patterns\nRepeat patterns and test tone...
870883,ladygaga,fashionofhislove,Lady Gaga,Lady Gaga,Fashion Of His Love,Fashion of His Love,both,[Verse 1]\nI never was the kind of girl that's...
850229,filter,catchafallingknife,Filter,Filter,Catch A Falling Knife,Catch a Falling Knife,both,[Verse 1: Richard Patrick]\nLife was just a sc...


#### Remaining Data
Not all data has been matched, but we still got around 360k matches which is quite a good base for our next tasks.

In [ ]:
no_matches_df = merge_df_norm[merge_df_norm['normalized_merge'] == 'left_only']
print(no_matches_df.shape)
no_matches_df.sample(20)[['artist_name_norm', 'track_name_norm', 'artist_name', 'artist_name_genius' , 'track_name', 'track_name_genius', 'normalized_merge', 'lyrics']]

(767302, 33)


,artist_name_norm,track_name_norm,artist_name,artist_name_genius,track_name,track_name_genius,normalized_merge,lyrics
693249,rusko,trupowwa,Rusko,NaN,Tru Powwa,NaN,left_only,NaN
160868,giljoe,farandwidefeatnkay,Gil Joe,NaN,Far and Wide (feat. Nkay),NaN,left_only,NaN
172710,bandlez,vacant,Bandlez,NaN,VACANT,NaN,left_only,NaN
654867,judygold,cnnthenewsandtheolympics,Judy Gold,NaN,"Cnn, The News and the Olympics",NaN,left_only,NaN
611769,drumagick,malandragem,Drumagick,NaN,Malandragem,NaN,left_only,NaN
321947,jamiekilstein,mikepence,Jamie Kilstein,NaN,Mike Pence,NaN,left_only,NaN
379754,vibie,tunaktunaktunlofiflip,VIBIE,NaN,Tunak Tunak Tun - Lofi Flip,NaN,left_only,NaN
768651,forrdomuido,todaboaaovivo,Forró do Muido,NaN,Toda Boa - Ao Vivo,NaN,left_only,NaN
334011,hemdale,miley,Hemdale,NaN,Miley,NaN,left_only,NaN
248579,francisagyei,wonnyawmemahuyesulive,Francis Agyei,NaN,Wonnyaw Me Mahu Yesu - Live,NaN,left_only,NaN


### Saving dataset

In [ ]:
if not os.path.exists("data"):
    os.makedirs("data")

In [ ]:
# Save the full merge as well
pd.concat(
    [
        raw_merge_df.drop(columns=["raw_merge"]).assign(merge_type="raw"), 
        normalized_merge_df.drop(columns=["normalized_merge"]).assign(merge_type="normalized"),
    ], 
    ignore_index=True
    ).to_parquet("data/tracks_merged.parquet")

In [ ]:
statistics_dict = {
    "spotify_df": spotify_df.shape[0],
    "genius_df": genius_df.shape[0],

    "raw_merge": raw_merge_df.shape[0],
    "unmatched_raw": raw_unmatched_spotify_df.shape[0],

    "unmatched_raw_notnan": unmatched_notnan.shape[0],
    "unmatched_raw_nan": normalized_merge_nan_removal_count,

    "norm_merge": normalized_merge_df.shape[0],
    "unmatched": no_matches_df.shape[0],
    
    "matched": normalized_merge_df.shape[0] + raw_merge_df.shape[0],
}

with open('data/statistics.pkl', 'wb') as f:
    pickle.dump(statistics_dict, f)

statistics_dict

{'spotify_df': 1159748,
 'genius_df': 5134668,
 'raw_merge': 231973,
 'unmatched_raw': 927775,
 'unmatched_raw_notnan': 898698,
 'unmatched_raw_nan': 29077,
 'norm_merge': 132963,
 'unmatched': 767302,
 'matched': 364936}